In [17]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

import shap

# Map x1-x23 to the official UCI variable names so the notebook is self-explanatory.
COLUMN_MAP = {
    "x1": "limit_bal",
    "x2": "sex",
    "x3": "education",
    "x4": "marriage",
    "x5": "age",
    "x6": "pay_0", 
    "x7": "pay_2", 
    "x8": "pay_3", 
    "x9": "pay_4",
    "x10": "pay_5", 
    "x11": "pay_6",
    "x12": "bill_amt1", 
    "x13": "bill_amt2", 
    "x14": "bill_amt3",
    "x15": "bill_amt4", 
    "x16": "bill_amt5", 
    "x17": "bill_amt6",
    "x18": "pay_amt1", 
    "x19": "pay_amt2", 
    "x20": "pay_amt3",
    "x21": "pay_amt4", 
    "x22": "pay_amt5", 
    "x23": "pay_amt6",
}

# fetch the dataset directly from OpenML (cached locally by sklearn after the first run)
raw = fetch_openml(data_id=42477, as_frame=True, parser="auto")
X = raw.data.rename(columns=COLUMN_MAP)
y = raw.target.astype(int)

CLASS_NAMES = ["No Default", "Default"]

# same split as Task 1/2: 80/20, stratified, fixed seed for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [18]:
models = {}

#Model 1: Logistic Regression
models["logistic_regression"] = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced").fit(X_train_scaled, y_train)

#Model 2: Random Forest
models["random_forest"] = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1).fit(X_train_scaled, y_train)

#Model 3: Gradient Boosting
gradient_boosting_sample_weight = compute_sample_weight("balanced", y_train)
models["gradient_boosting"] = GradientBoostingClassifier(random_state=42).fit(X_train_scaled, y_train, sample_weight=gradient_boosting_sample_weight)

performance_rows = []

for name, model in models.items():
    y_prediction = model.predict(X_test_scaled)
    y_probability = model.predict_proba(X_test_scaled)[:, 1]
    performance_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_prediction),
        "auc": roc_auc_score(y_test, y_probability),
    })

performance_df = pd.DataFrame(performance_rows)
performance_df

,model,accuracy,auc
0,logistic_regression,0.679667,0.708115
1,random_forest,0.800833,0.761115
2,gradient_boosting,0.764833,0.779242


In [19]:
N_SAMPLES = 30
TOP_K = 10
LIME_REF_SEED = 42
LIME_SEEDS = [0, 1, 2, 3, 4, 42, 99]

# Same random_state as Task 2 -> same 30 clients
sample_df = X_test_scaled.sample(N_SAMPLES, random_state=42)
sample_indices = sample_df.index.tolist()
print(f"Selected {len(sample_indices)} test clients \n")


def positive_class_shap_values(explainer, X):
    explanation = explainer(X)
    values = explanation.values
    if values.ndim == 3:
        return values[:, :, 1]
    return values

shap_explainers = {
    "logistic_regression": shap.LinearExplainer(models["logistic_regression"], X_train_scaled),
    "random_forest": shap.TreeExplainer(models["random_forest"]),
    "gradient_boosting": shap.TreeExplainer(models["gradient_boosting"]),
}

shap_values_by_model = {}
for name, explainer in shap_explainers.items():
    values = positive_class_shap_values(explainer, sample_df)
    shap_values_by_model[name] = pd.DataFrame(values, columns=X.columns, index=sample_indices)

# Sanity check: top feature for the first client, per model
for name, values_df in shap_values_by_model.items():
    top_feature = values_df.iloc[0].abs().idxmax()
    print(f"{name}: top feature for first client = {top_feature}")

Background dataset has 24000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=24000 when initializing the masker.


Selected 30 test clients 

logistic_regression: top feature for first client = bill_amt1
random_forest: top feature for first client = pay_4
gradient_boosting: top feature for first client = pay_0


In [20]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=X.columns.tolist(),
    class_names=CLASS_NAMES,
    mode="classification",
    random_state=LIME_REF_SEED,
)

# feature isim + ağırlık (işaret dahil) döndürüyor
def lime_top_feature_weights(explanation, feature_names, k=TOP_K):
    weights = dict(explanation.as_map()[1])
    ordered = sorted(weights.items(), key=lambda item: abs(item[1]), reverse=True)
    return {feature_names[idx]: weight for idx, weight in ordered[:k]}
def predict_proba_with_names(model):
    # LIME calls this with a raw numpy array; wrap it back into a DataFrame, with original column names
    return lambda arr: model.predict_proba(pd.DataFrame(arr, columns=X.columns))

lime_top_features_by_model = {}

for name, model in models.items():
    predict_fn = predict_proba_with_names(model)
    rows = []
    for client_id in sample_indices:
        explanation = lime_explainer.explain_instance(
            sample_df.loc[client_id].values,
            predict_fn,
            num_features=TOP_K,
        )
        rows.append({
            "client_id": client_id,
            "weights": lime_top_feature_weights(explanation, X.columns.tolist()),
        })
    lime_top_features_by_model[name] = pd.DataFrame(rows).set_index("client_id")

for name, df in lime_top_features_by_model.items():
    print(f"{name}: top feature for first client = {next(iter(df.iloc[0]['weights']))}")

logistic_regression: top feature for first client = bill_amt1
random_forest: top feature for first client = pay_0
gradient_boosting: top feature for first client = pay_0


In [ ]:
def top_k_features_with_sign(shap_row, k=TOP_K):
    # bir client'in SHAP değerlerinden en önemli k tanesini, işaretiyle birlikte al
    top = shap_row.abs().nlargest(k)
    return {feature: np.sign(shap_row[feature]) for feature in top.index}

def overlap_at_k(shap_features, lime_features, k=TOP_K):
    return len(set(shap_features) & set(lime_features)) / k

def sign_agreement(shap_signs, lime_signs):
    shared = set(shap_signs) & set(lime_signs)
    if not shared:
        return np.nan
    return np.mean([shap_signs[f] == lime_signs[f] for f in shared])

agreement_rows = []
for model_name in models:
    shap_df = shap_values_by_model[model_name]
    lime_df = lime_top_features_by_model[model_name]

    for client_id in sample_indices:
        shap_signs = top_k_features_with_sign(shap_df.loc[client_id])
        lime_weights = lime_df.loc[client_id, "weights"]
        lime_signs = {f: np.sign(w) for f, w in lime_weights.items()}

        agreement_rows.append({
            "model": model_name,
            "overlap_at_k": overlap_at_k(shap_signs.keys(), lime_signs.keys()),
            "sign_agreement": sign_agreement(shap_signs, lime_signs),
        })

agreement_df = pd.DataFrame(agreement_rows)

# 30 client üzerinden ortalama -> model başına tek satır
top_feature_agreement_table = agreement_df.groupby("model")[["overlap_at_k", "sign_agreement"]].mean()
top_feature_agreement_table


#overlap_at_k: İki farklı açıklama yöntemi aynı “önemli” değişkenleri mi gösteriyor? 
#Yüksek overlap = “SHAP ve LIME aynı hikâyeyi anlatıyor” demek. 
#Düşük overlap = birine güvenmeden önce dikkat.

#sign_agreement: Sadece “aynı özellik” yetmez; aynı yönde mi etkiliyor ona bakıyor. 
#Overlap yüksek ama sign_agreement düşükse: “Doğru değişkenleri buluyorlar ama biri artırıcı, biri azaltıcı der" ciddi güvenilirlik sorunu yaratır.

,overlap_at_k,sign_agreement
model,,
gradient_boosting,0.660000,0.945384
logistic_regression,0.793333,1.000000
random_forest,0.636667,0.941627


In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

stability_rows = []

for seed in LIME_SEEDS:
    if seed == LIME_REF_SEED:
        continue  # referansı kendisiyle karşılaştırmanın anlamı yok, hep 1.0 çıkar

    seed_explainer = LimeTabularExplainer(
        training_data=X_train_scaled.values,
        feature_names=X.columns.tolist(),
        class_names=CLASS_NAMES,
        mode="classification",
        random_state=seed,
    )

    for name, model in models.items():
        predict_fn = predict_proba_with_names(model)
        ref_df = lime_top_features_by_model[name]

        for client_id in sample_indices:
            explanation = seed_explainer.explain_instance(
                sample_df.loc[client_id].values,
                predict_fn,
                num_features=TOP_K,
            )
            seed_weights = lime_top_feature_weights(explanation, X.columns.tolist())
            ref_weights = ref_df.loc[client_id, "weights"]

            stability_rows.append({
                "model": name,
                "seed": seed,
                "jaccard": jaccard(set(seed_weights.keys()), set(ref_weights.keys())),
            })

stability_df = pd.DataFrame(stability_rows)

# 6 seed x 30 client üzerinden ortalama -> model başına tek satır
lime_stability_table = stability_df.groupby("model")["jaccard"].mean().to_frame("avg_jaccard")
lime_stability_table


#avg_jaccard — LIME kendi kendine tutarlı mı?
#avg_jaccard: LIME rastgele örnekleme kullandığı için aynı müşteriye her seferinde aynı açıklamayı veriyor mu? 
#Yüksek Jaccard = stabil; düşük = “seed değiştirince farklı özellikler çıkıyor” -> açıklamaya göre zayıflar

,avg_jaccard
model,
gradient_boosting,0.703478
logistic_regression,0.790084
random_forest,0.673932


Overall assessment: Logistic Regression appears to be the best performer in terms of both SHAP–LIME alignment and LIME stability. Although Random Forest achieves high accuracy, its explanations are less consistent—signaling that 
"the model makes good predictions, but explaining the 'why' is difficult." These metrics exist specifically to compare explanation reliability independently of model performance.


Toplu yorum: Logistic Regression hem SHAP–LIME uyumunda hem LIME stabilitesinde en iyi görünüyor. Random Forest accuracy’si yüksek olsa da açıklamalar daha az tutarlı yani 
“model iyi tahmin ediyor ama nedenini anlatmak zor” sinyali veriyor. Bu metrikler tam olarak model performansından bağımsız olarak açıklama güvenilirliğini karşılaştırmak için var.

In [ ]:
PERTURBATION_SCALE = 0.1  # küçük gürültü: features zaten scaled

rng = np.random.default_rng(42)

def perturb_row(row):
    noise = rng.normal(loc=0.0, scale=PERTURBATION_SCALE, size=len(row))
    return row + noise

robustness_rows = []

for name, model in models.items():
    shap_df = shap_values_by_model[name]
    lime_df = lime_top_features_by_model[name]
    explainer = shap_explainers[name]
    predict_fn = predict_proba_with_names(model)

    for client_id in sample_indices:
        original_row = sample_df.loc[client_id]
        perturbed_row = perturb_row(original_row)
        perturbed_df = perturbed_row.to_frame().T
        perturbed_df.columns = X.columns

        # SHAP: orijinal client vs hafifçe değiştirilmiş client
        original_shap_top = set(top_k_features_with_sign(shap_df.loc[client_id]).keys())
        perturbed_shap_values = positive_class_shap_values(explainer, perturbed_df)[0]
        perturbed_shap_top = set(top_k_features_with_sign(pd.Series(perturbed_shap_values, index=X.columns)).keys())
        shap_robustness = jaccard(original_shap_top, perturbed_shap_top)

        # LIME: aynı referans seed (42) kullanıyoruz -> ölçtüğümüz şey sadece "input değişimine duyarlılık", LIME'ın kendi rastgeleliği değil
        original_lime_top = set(lime_df.loc[client_id, "weights"].keys())
        explanation = lime_explainer.explain_instance(perturbed_row.values, predict_fn, num_features=TOP_K)
        perturbed_lime_top = set(lime_top_feature_weights(explanation, X.columns.tolist()).keys())
        lime_robustness = jaccard(original_lime_top, perturbed_lime_top)

        robustness_rows.append({
            "model": name,
            "shap_robustness": shap_robustness,
            "lime_robustness": lime_robustness,
        })

robustness_df = pd.DataFrame(robustness_rows)
robustness_table = robustness_df.groupby("model")[["shap_robustness", "lime_robustness"]].mean()
robustness_table


#shap_robustness: Gürültü eklenince SHAP top-10 ne kadar korunuyor
#lime_robustness: Gürültü eklenince LIME top-10 ne kadar korunuyor

#SHAP genel olarak LIME’dan daha robust (gürültüye daha dayanıklı). LR yine en iyi.

,shap_robustness,lime_robustness
model,,
gradient_boosting,0.681341,0.534796
logistic_regression,0.862626,0.626096
random_forest,0.610312,0.568265


In [24]:
reliability_table = pd.concat([
    top_feature_agreement_table,
    lime_stability_table,
    robustness_table,
], axis=1)

reliability_table["reliability_score"] = reliability_table.mean(axis=1)
reliability_table.sort_values("reliability_score", ascending=False)

,overlap_at_k,sign_agreement,avg_jaccard,shap_robustness,lime_robustness,reliability_score
model,,,,,,
logistic_regression,0.793333,1.000000,0.790084,0.862626,0.626096,0.814428
gradient_boosting,0.660000,0.945384,0.703478,0.681341,0.534796,0.705000
random_forest,0.636667,0.941627,0.673932,0.610312,0.568265,0.686160


Summary:

Across the three models trained on the same credit-default dataset, Logistic Regression produced the most reliable SHAP/LIME explanations (reliability score 0.81), clearly ahead of Gradient Boosting (0.71) and Random Forest (0.69). This holds on every axis measured SHAP–LIME agreement, LIME stability across random seeds, and robustness to small input perturbations. Random Forest had the highest raw accuracy (0.80) but the least reliable explanations predictive performance and explanation reliability move in opposite directions here. Across all three models, SHAP was consistently more robust to input perturbations than LIME.


Aynı kredi-default veri setinde eğitilen 3 model arasında, Logistic Regression en güvenilir SHAP/LIME açıklamalarını üretti (reliability score 0.81) Gradient Boosting'in (0.71) ve Random Forest'ın (0.69) açık farkla önünde. Bu sonuç ölçtüğümüz her eksende geçerli: SHAP–LIME uyumu, LIME'ın seed'ler arası kararlılığı, girdi değişimine karşı dayanıklılık. Random Forest en yüksek ham accuracy'ye sahip (0.80) ama açıklamaları en az güvenilir olan model. Tahmin performansı ile açıklama güvenilirliği burada ters yönde ilerliyor. Üç modelde de SHAP, girdi dengesindeki bozulmaya karşı LIME'dan tutarlı biçimde daha dayanıklı çıktı.